In [ ]:
!pip install transformers torch sentencepiece datasets indic-nlp-library

import torch
from transformers import AutoTokenizer, LlamaTokenizer
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import re
import json

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Dependencies imported successfully")

In [ ]:
# Load Llama 3.1 8B tokenizer
model_name = "meta-llama/Llama-3.1-8B"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"✓ Loaded tokenizer: {model_name}")
    print(f"Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"Tokenizer type: {type(tokenizer).__name__}")
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("\nNote: You may need to accept the license on HuggingFace and login:")
    print("Run: huggingface-cli login")

In [ ]:
from huggingface_hub import login
login(token="hf_REDACTED_ROTATE_THIS_TOKEN")

In [ ]:
# Load Gemma 2 9B tokenizer
model_name = "google/gemma-2-9b"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"✓ Loaded tokenizer: {model_name}")
    print(f"Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"Tokenizer type: {type(tokenizer).__name__}")
    
    # Gemma uses SentencePiece tokenizer
    print(f"\nTokenizer class: {tokenizer.__class__.__name__}")
    
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("\nNote: You may need to accept the license on HuggingFace and login:")
    print("Run: huggingface-cli login")
    print("\nOr install: pip install sentencepiece")

In [ ]:
# Sample Tamil corpus - you'll expand this with your own data
# This includes different types of Tamil text

tamil_corpus = [
    # Simple sentences
    "நான் பள்ளிக்குச் செல்கிறேன்",
    "அவன் புத்தகம் படிக்கிறான்",
    "அவள் சாப்பிடுகிறாள்",
    
    # Mathematical/reasoning text
    "இரண்டு கூட்டல் மூன்று சமம் ஐந்து",
    "2 + 3 = 5",
    "பத்து பெருக்கல் பத்து சமம் நூறு",
    
    # Complex sentences with morphology
    "மாணவர்கள் விளையாடிக்கொண்டிருக்கிறார்கள்",
    "அவர்கள் சென்றுகொண்டிருந்தார்கள்",
    
    # Questions
    "இது என்ன?",
    "எங்கே போகிறாய்?",
    "ஏன் வரவில்லை?",
    
    # Numbers and mixed content
    "2024 ஆம் ஆண்டு",
    "விலை 100 ரூபாய்",
    
    # Longer text
    "தமிழ் மொழி உலகின் பழமையான மொழிகளில் ஒன்றாகும்",
    "கணிதம் என்பது எண்களின் அறிவியல் ஆகும்",
]

# You can also load from file
# with open('tamil_corpus.txt', 'r', encoding='utf-8') as f:
#     tamil_corpus = f.readlines()

print(f"✓ Loaded {len(tamil_corpus)} Tamil sentences")
print("\nSample sentences:")
for i, sent in enumerate(tamil_corpus[:3]):
    print(f"{i+1}. {sent}")

In [ ]:
def count_tamil_words(text: str) -> int:
    """
    Count words in Tamil text
    Words are separated by spaces or punctuation
    """
    # Remove punctuation and split
    words = re.findall(r'[\u0B80-\u0BFF]+', text)
    return len(words)

def count_tamil_characters(text: str) -> int:
    """
    Count Tamil characters (excluding spaces and punctuation)
    """
    tamil_chars = re.findall(r'[\u0B80-\u0BFF]', text)
    return len(tamil_chars)

def extract_tamil_words(text: str) -> List[str]:
    """
    Extract Tamil words from text
    """
    return re.findall(r'[\u0B80-\u0BFF]+', text)

def get_grapheme_clusters(text: str) -> List[str]:
    """
    Basic Tamil grapheme cluster extraction
    A grapheme = base character + modifiers
    """
    # This is a simplified version - you can improve with proper grapheme library
    graphemes = []
    tamil_chars = list(text)
    
    i = 0
    while i < len(tamil_chars):
        cluster = tamil_chars[i]
        # Check if next char is a dependent vowel sign (U+0BBE-U+0BCD)
        while i + 1 < len(tamil_chars) and '\u0BBE' <= tamil_chars[i + 1] <= '\u0BCD':
            i += 1
            cluster += tamil_chars[i]
        graphemes.append(cluster)
        i += 1
    
    return graphemes

# Test the functions
test_text = "நான் படிக்கிறேன்"
print(f"Test text: {test_text}")
print(f"Words: {count_tamil_words(test_text)}")
print(f"Characters: {count_tamil_characters(test_text)}")
print(f"Words extracted: {extract_tamil_words(test_text)}")
print(f"Graphemes: {get_grapheme_clusters(test_text)}")

In [ ]:
def analyze_tokenization(tokenizer, text: str) -> Dict:
    """
    Analyze how a tokenizer handles a given text
    """
    # Tokenize
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    
    # Get token strings (decode each token separately)
    token_strings = [tokenizer.decode([tid]) for tid in token_ids]
    
    # Count words and characters
    num_words = count_tamil_words(text)
    num_chars = count_tamil_characters(text)
    num_tokens = len(tokens)
    
    return {
        'text': text,
        'tokens': tokens,
        'token_ids': token_ids,
        'token_strings': token_strings,
        'num_tokens': num_tokens,
        'num_words': num_words,
        'num_chars': num_chars,
        'text_length': len(text)
    }

def visualize_tokenization(tokenizer, text: str):
    """
    Visualize how text is tokenized
    """
    result = analyze_tokenization(tokenizer, text)
    
    print(f"Original text: {text}")
    print(f"Length: {result['text_length']} characters")
    print(f"Tamil words: {result['num_words']}")
    print(f"Tamil characters: {result['num_chars']}")
    print(f"Tokens: {result['num_tokens']}")
    print(f"\nTokens: {result['tokens']}")
    print(f"\nToken strings: {result['token_strings']}")
    print("-" * 80)

# Test on a few examples
print("TOKENIZATION EXAMPLES:\n")
for text in tamil_corpus[:3]:
    visualize_tokenization(tokenizer, text)
    print()

In [ ]:
def calculate_token_fertility(tokenizer, corpus: List[str]) -> Dict:
    """
    Token Fertility = Average number of tokens per word
    Lower fertility = more efficient
    """
    total_tokens = 0
    total_words = 0
    fertility_per_sentence = []
    
    for text in corpus:
        result = analyze_tokenization(tokenizer, text)
        total_tokens += result['num_tokens']
        total_words += result['num_words']
        
        if result['num_words'] > 0:
            fertility = result['num_tokens'] / result['num_words']
            fertility_per_sentence.append(fertility)
    
    avg_fertility = total_tokens / total_words if total_words > 0 else 0
    
    return {
        'average_fertility': avg_fertility,
        'total_tokens': total_tokens,
        'total_words': total_words,
        'fertility_per_sentence': fertility_per_sentence,
        'min_fertility': min(fertility_per_sentence) if fertility_per_sentence else 0,
        'max_fertility': max(fertility_per_sentence) if fertility_per_sentence else 0,
        'std_fertility': np.std(fertility_per_sentence) if fertility_per_sentence else 0
    }

# Calculate fertility
fertility_results = calculate_token_fertility(tokenizer, tamil_corpus)

print("=" * 80)
print("TOKEN FERTILITY ANALYSIS")
print("=" * 80)
print(f"\nAverage Token Fertility: {fertility_results['average_fertility']:.3f}")
print(f"Interpretation: On average, each Tamil word is split into {fertility_results['average_fertility']:.2f} tokens")
print(f"\nTotal tokens generated: {fertility_results['total_tokens']}")
print(f"Total Tamil words: {fertility_results['total_words']}")
print(f"\nRange: {fertility_results['min_fertility']:.2f} - {fertility_results['max_fertility']:.2f}")
print(f"Standard deviation: {fertility_results['std_fertility']:.3f}")

# Visualize
plt.figure(figsize=(10, 5))
plt.hist(fertility_results['fertility_per_sentence'], bins=20, edgecolor='black', alpha=0.7)
plt.axvline(fertility_results['average_fertility'], color='red', linestyle='--', 
            label=f'Average: {fertility_results["average_fertility"]:.2f}')
plt.xlabel('Token Fertility (tokens per word)')
plt.ylabel('Frequency')
plt.title('Distribution of Token Fertility Across Sentences')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def calculate_normalized_sequence_length(tokenizer, corpus: List[str]) -> Dict:
    """
    NSL = Ratio of tokenized length to original text length
    Measures compression efficiency
    """
    nsl_values = []
    
    for text in corpus:
        result = analyze_tokenization(tokenizer, text)
        
        # NSL can be measured in different ways:
        # 1. tokens / characters
        nsl_chars = result['num_tokens'] / result['num_chars'] if result['num_chars'] > 0 else 0
        
        # 2. tokens / text_length (including spaces)
        nsl_text = result['num_tokens'] / result['text_length'] if result['text_length'] > 0 else 0
        
        nsl_values.append({
            'text': text,
            'nsl_chars': nsl_chars,
            'nsl_text': nsl_text,
            'num_tokens': result['num_tokens'],
            'num_chars': result['num_chars'],
            'text_length': result['text_length']
        })
    
    avg_nsl_chars = np.mean([v['nsl_chars'] for v in nsl_values])
    avg_nsl_text = np.mean([v['nsl_text'] for v in nsl_values])
    
    return {
        'nsl_values': nsl_values,
        'avg_nsl_chars': avg_nsl_chars,
        'avg_nsl_text': avg_nsl_text,
        'std_nsl_chars': np.std([v['nsl_chars'] for v in nsl_values]),
        'std_nsl_text': np.std([v['nsl_text'] for v in nsl_values])
    }

# Calculate NSL
nsl_results = calculate_normalized_sequence_length(tokenizer, tamil_corpus)

print("=" * 80)
print("NORMALIZED SEQUENCE LENGTH (NSL) ANALYSIS")
print("=" * 80)
print(f"\nAverage NSL (tokens/Tamil characters): {nsl_results['avg_nsl_chars']:.3f}")
print(f"Average NSL (tokens/text length): {nsl_results['avg_nsl_text']:.3f}")
print(f"\nInterpretation:")
print(f"  - On average, 1 Tamil character requires {nsl_results['avg_nsl_chars']:.2f} tokens")
print(f"  - For every character of original text, we get {nsl_results['avg_nsl_text']:.2f} tokens")

# Create comparison table
print("\nPer-sentence breakdown:")
df_nsl = pd.DataFrame(nsl_results['nsl_values'])
df_nsl['text_short'] = df_nsl['text'].str[:30] + '...'
print(df_nsl[['text_short', 'num_tokens', 'num_chars', 'nsl_chars']].to_string(index=False))

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist([v['nsl_chars'] for v in nsl_results['nsl_values']], bins=15, edgecolor='black', alpha=0.7)
ax1.axvline(nsl_results['avg_nsl_chars'], color='red', linestyle='--', 
            label=f'Avg: {nsl_results["avg_nsl_chars"]:.2f}')
ax1.set_xlabel('NSL (tokens per Tamil character)')
ax1.set_ylabel('Frequency')
ax1.set_title('NSL Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.scatter(range(len(nsl_results['nsl_values'])), 
            [v['nsl_chars'] for v in nsl_results['nsl_values']], alpha=0.6)
ax2.axhline(nsl_results['avg_nsl_chars'], color='red', linestyle='--', 
            label=f'Average: {nsl_results["avg_nsl_chars"]:.2f}')
ax2.set_xlabel('Sentence Index')
ax2.set_ylabel('NSL')
ax2.set_title('NSL Across Sentences')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def calculate_fragmentation_rate(tokenizer, corpus: List[str]) -> Dict:
    """
    Fragmentation Rate = How much are Tamil words split?
    Measures: tokens_per_word > 1 (fragmentation)
    """
    fragmentation_data = []
    
    for text in corpus:
        tamil_words = extract_tamil_words(text)
        
        for word in tamil_words:
            tokens = tokenizer.tokenize(word)
            num_tokens = len(tokens)
            is_fragmented = num_tokens > 1
            
            fragmentation_data.append({
                'word': word,
                'num_tokens': num_tokens,
                'is_fragmented': is_fragmented,
                'tokens': tokens
            })
    
    total_words = len(fragmentation_data)
    fragmented_words = sum(1 for d in fragmentation_data if d['is_fragmented'])
    fragmentation_rate = fragmented_words / total_words if total_words > 0 else 0
    
    # Calculate average fragmentation degree
    avg_tokens_per_word = np.mean([d['num_tokens'] for d in fragmentation_data])
    
    # Find most fragmented words
    most_fragmented = sorted(fragmentation_data, key=lambda x: x['num_tokens'], reverse=True)[:10]
    
    return {
        'fragmentation_rate': fragmentation_rate,
        'total_words': total_words,
        'fragmented_words': fragmented_words,
        'unfragmented_words': total_words - fragmented_words,
        'avg_tokens_per_word': avg_tokens_per_word,
        'fragmentation_data': fragmentation_data,
        'most_fragmented': most_fragmented
    }

# Calculate fragmentation
frag_results = calculate_fragmentation_rate(tokenizer, tamil_corpus)

print("=" * 80)
print("FRAGMENTATION RATE ANALYSIS")
print("=" * 80)
print(f"\nFragmentation Rate: {frag_results['fragmentation_rate']:.1%}")
print(f"Interpretation: {frag_results['fragmentation_rate']*100:.1f}% of Tamil words are split into multiple tokens")
print(f"\nTotal Tamil words analyzed: {frag_results['total_words']}")
print(f"Fragmented words: {frag_results['fragmented_words']}")
print(f"Kept as single token: {frag_results['unfragmented_words']}")
print(f"Average tokens per word: {frag_results['avg_tokens_per_word']:.2f}")

print("\n" + "=" * 80)
print("MOST FRAGMENTED WORDS:")
print("=" * 80)
for i, item in enumerate(frag_results['most_fragmented'][:10], 1):
    print(f"{i}. '{item['word']}' → {item['num_tokens']} tokens: {item['tokens']}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: fragmented vs unfragmented
labels = ['Fragmented\n(>1 token)', 'Single Token']
sizes = [frag_results['fragmented_words'], frag_results['unfragmented_words']]
colors = ['#ff9999', '#66b3ff']
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Word Fragmentation Distribution')

# Histogram of tokens per word
tokens_per_word = [d['num_tokens'] for d in frag_results['fragmentation_data']]
ax2.hist(tokens_per_word, bins=range(1, max(tokens_per_word)+2), edgecolor='black', alpha=0.7)
ax2.axvline(frag_results['avg_tokens_per_word'], color='red', linestyle='--',
            label=f'Avg: {frag_results["avg_tokens_per_word"]:.2f}')
ax2.set_xlabel('Number of Tokens per Word')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Tokens per Word')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def analyze_morphological_fragmentation(tokenizer, corpus: List[str]) -> Dict:
    """
    Analyze how morphologically complex words are fragmented
    Focus on words with suffixes/prefixes
    """
    
    # Common Tamil suffixes (just examples - expand this)
    common_suffixes = [
        'கிறான்', 'கிறாள்', 'கிறார்கள்', 'கிறேன்',  # present tense
        'ஆன்', 'ாள்', 'ார்கள்',  # past tense markers
        'ுகிறான்', 'டுகிறான்', 'ந்து',  # various verb forms
    ]
    
    morphologically_complex = []
    
    for text in corpus:
        tamil_words = extract_tamil_words(text)
        
        for word in tamil_words:
            # Check if word has common suffix
            has_suffix = any(word.endswith(suf) for suf in common_suffixes)
            
            if has_suffix or len(word) > 8:  # Long words likely have morphology
                tokens = tokenizer.tokenize(word)
                
                morphologically_complex.append({
                    'word': word,
                    'length': len(word),
                    'num_tokens': len(tokens),
                    'tokens': tokens,
                    'has_known_suffix': has_suffix
                })
    
    if not morphologically_complex:
        return {'message': 'No morphologically complex words found in corpus'}
    
    avg_fragmentation = np.mean([w['num_tokens'] for w in morphologically_complex])
    
    return {
        'num_complex_words': len(morphologically_complex),
        'avg_fragmentation': avg_fragmentation,
        'complex_words': morphologically_complex
    }

# Analyze morphological fragmentation
morph_results = analyze_morphological_fragmentation(tokenizer, tamil_corpus)

if 'message' not in morph_results:
    print("=" * 80)
    print("MORPHOLOGICAL FRAGMENTATION ANALYSIS")
    print("=" * 80)
    print(f"\nMorphologically complex words found: {morph_results['num_complex_words']}")
    print(f"Average tokens per complex word: {morph_results['avg_fragmentation']:.2f}")
    
    print("\nExamples of morphological fragmentation:")
    for i, item in enumerate(morph_results['complex_words'][:10], 1):
        suffix_marker = " [has suffix]" if item['has_known_suffix'] else ""
        print(f"{i}. '{item['word']}'{suffix_marker}")
        print(f"   → {item['num_tokens']} tokens: {item['tokens']}")
else:
    print(morph_results['message'])

In [ ]:
def analyze_number_tokenization(tokenizer) -> Dict:
    """
    Analyze how numbers are tokenized - critical for math reasoning
    """
    
    test_numbers = [
        "1", "2", "5", "10", "25", "100", "1000", "2024",
        "3.14", "0.5", "123.456",
        "1+2", "10-5", "2*3", "10/2", "2+3=5",
        "10%", "₹100", "$50"
    ]
    
    number_analysis = []
    
    for num in test_numbers:
        tokens = tokenizer.tokenize(num)
        num_tokens = len(tokens)
        
        number_analysis.append({
            'number': num,
            'num_tokens': num_tokens,
            'tokens': tokens,
            'is_single_token': num_tokens == 1
        })
    
    single_token_rate = sum(1 for n in number_analysis if n['is_single_token']) / len(number_analysis)
    
    return {
        'number_analysis': number_analysis,
        'single_token_rate': single_token_rate,
        'avg_tokens_per_number': np.mean([n['num_tokens'] for n in number_analysis])
    }

# Analyze numbers
num_results = analyze_number_tokenization(tokenizer)

print("=" * 80)
print("NUMBER TOKENIZATION ANALYSIS")
print("=" * 80)
print(f"\nSingle token rate: {num_results['single_token_rate']:.1%}")
print(f"Average tokens per number: {num_results['avg_tokens_per_number']:.2f}")

print("\nDetailed breakdown:")
for item in num_results['number_analysis']:
    status = "✓ Single token" if item['is_single_token'] else f"✗ Split into {item['num_tokens']}"
    print(f"{item['number']:10s} → {status:20s} {item['tokens']}")

In [ ]:
def generate_summary_report(model_name, fertility_results, nsl_results, 
                           frag_results, num_results, corpus_size):
    """
    Generate comprehensive summary of all metrics
    """
    
    print("=" * 80)
    print(f"TOKENIZATION ANALYSIS SUMMARY - {model_name}")
    print("=" * 80)
    
    print(f"\nCorpus: {corpus_size} Tamil sentences")
    print(f"Tokenizer vocabulary size: {tokenizer.vocab_size:,}")
    
    print("\n" + "-" * 80)
    print("1. TOKEN FERTILITY")
    print("-" * 80)
    print(f"   Average: {fertility_results['average_fertility']:.3f} tokens/word")
    print(f"   Range: {fertility_results['min_fertility']:.2f} - {fertility_results['max_fertility']:.2f}")
    print(f"   Interpretation: Tamil words require {fertility_results['average_fertility']:.1f}x more tokens than English baseline (1.0)")
    
    print("\n" + "-" * 80)
    print("2. NORMALIZED SEQUENCE LENGTH")
    print("-" * 80)
    print(f"   NSL (tokens/character): {nsl_results['avg_nsl_chars']:.3f}")
    print(f"   Interpretation: Each Tamil character maps to {nsl_results['avg_nsl_chars']:.2f} tokens on average")
    
    print("\n" + "-" * 80)
    print("3. FRAGMENTATION RATE")
    print("-" * 80)
    print(f"   Fragmentation: {frag_results['fragmentation_rate']:.1%} of words split")
    print(f"   Average tokens/word: {frag_results['avg_tokens_per_word']:.2f}")
    print(f"   Single token: {frag_results['unfragmented_words']}/{frag_results['total_words']} words")
    
    print("\n" + "-" * 80)
    print("4. NUMBER HANDLING")
    print("-" * 80)
    print(f"   Single token rate: {num_results['single_token_rate']:.1%}")
    print(f"   Average tokens/number: {num_results['avg_tokens_per_number']:.2f}")
    
    print("\n" + "=" * 80)
    print("KEY FINDINGS:")
    print("=" * 80)
    
    # Generate insights
    if fertility_results['average_fertility'] > 2.5:
        print(f"⚠ HIGH FERTILITY: Tamil words require {fertility_results['average_fertility']:.1f}x tokens (inefficient)")
    elif fertility_results['average_fertility'] < 1.5:
        print(f"✓ GOOD FERTILITY: Efficient tokenization ({fertility_results['average_fertility']:.1f} tokens/word)")
    
    if frag_results['fragmentation_rate'] > 0.7:
        print(f"⚠ HIGH FRAGMENTATION: {frag_results['fragmentation_rate']:.0%} of words split (may hurt reasoning)")
    
    if num_results['single_token_rate'] < 0.5:
        print(f"⚠ POOR NUMBER HANDLING: Only {num_results['single_token_rate']:.0%} of numbers as single tokens")
    else:
        print(f"✓ GOOD NUMBER HANDLING: {num_results['single_token_rate']:.0%} of numbers as single tokens")
    
    print("=" * 80)

# Generate report
generate_summary_report(
    model_name=model_name,
    fertility_results=fertility_results,
    nsl_results=nsl_results,
    frag_results=frag_results,
    num_results=num_results,
    corpus_size=len(tamil_corpus)
)

In [ ]:
# Prepare results for export
results_export = {
    'model_name': model_name,
    'vocabulary_size': tokenizer.vocab_size,
    'corpus_size': len(tamil_corpus),
    
    'metrics': {
        'token_fertility': {
            'average': float(fertility_results['average_fertility']),
            'min': float(fertility_results['min_fertility']),
            'max': float(fertility_results['max_fertility']),
            'std': float(fertility_results['std_fertility']),
            'total_tokens': fertility_results['total_tokens'],
            'total_words': fertility_results['total_words']
        },
        
        'normalized_sequence_length': {
            'avg_nsl_chars': float(nsl_results['avg_nsl_chars']),
            'avg_nsl_text': float(nsl_results['avg_nsl_text']),
            'std_nsl_chars': float(nsl_results['std_nsl_chars'])
        },
        
        'fragmentation': {
            'fragmentation_rate': float(frag_results['fragmentation_rate']),
            'total_words': frag_results['total_words'],
            'fragmented_words': frag_results['fragmented_words'],
            'avg_tokens_per_word': float(frag_results['avg_tokens_per_word']),
            'most_fragmented_examples': [
                {'word': w['word'], 'num_tokens': w['num_tokens'], 'tokens': w['tokens']}
                for w in frag_results['most_fragmented'][:5]
            ]
        },
        
        'number_handling': {
            'single_token_rate': float(num_results['single_token_rate']),
            'avg_tokens_per_number': float(num_results['avg_tokens_per_number'])
        }
    }
}

# Save to JSON
output_filename = f"tokenization_analysis_{model_name.replace('/', '_')}.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_export, f, ensure_ascii=False, indent=2)

print(f"✓ Results saved to: {output_filename}")

In [ ]:
# Create summary DataFrame
summary_data = {
    'Model': [model_name],
    'Vocab Size': [tokenizer.vocab_size],
    'Token Fertility': [f"{fertility_results['average_fertility']:.3f}"],
    'NSL (tokens/char)': [f"{nsl_results['avg_nsl_chars']:.3f}"],
    'Fragmentation Rate': [f"{frag_results['fragmentation_rate']:.1%}"],
    'Avg Tokens/Word': [f"{frag_results['avg_tokens_per_word']:.2f}"],
    'Number Single Token %': [f"{num_results['single_token_rate']:.1%}"]
}

df_summary = pd.DataFrame(summary_data)

print("SUMMARY TABLE:")
print(df_summary.to_string(index=False))

# Save to CSV for later comparison with other models
df_summary.to_csv('tokenization_summary.csv', index=False)
print("\n✓ Summary saved to: tokenization_summary.csv")

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Token Fertility
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(['Fertility'], [fertility_results['average_fertility']], color='skyblue', edgecolor='black')
ax1.set_ylabel('Tokens per Word')
ax1.set_title('Token Fertility')
ax1.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Ideal (1.0)')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# 2. NSL
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(['NSL'], [nsl_results['avg_nsl_chars']], color='lightcoral', edgecolor='black')
ax2.set_ylabel('Tokens per Character')
ax2.set_title('Normalized Sequence Length')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Fragmentation pie
ax3 = fig.add_subplot(gs[0, 2])
labels = [f'Fragmented\n({frag_results["fragmentation_rate"]:.0%})', 
          f'Single Token\n({1-frag_results["fragmentation_rate"]:.0%})']
sizes = [frag_results['fragmented_words'], frag_results['unfragmented_words']]
ax3.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['#ff9999', '#66b3ff'])
ax3.set_title('Word Fragmentation')

# 4. Fertility distribution
ax4 = fig.add_subplot(gs[1, :2])
ax4.hist(fertility_results['fertility_per_sentence'], bins=20, edgecolor='black', alpha=0.7, color='skyblue')
ax4.axvline(fertility_results['average_fertility'], color='red', linestyle='--', linewidth=2,
            label=f'Mean: {fertility_results["average_fertility"]:.2f}')
ax4.set_xlabel('Token Fertility')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Token Fertility Across Sentences')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Number handling
ax5 = fig.add_subplot(gs[1, 2])
number_types = ['Single\nToken', 'Split']
number_counts = [
    sum(1 for n in num_results['number_analysis'] if n['is_single_token']),
    sum(1 for n in num_results['number_analysis'] if not n['is_single_token'])
]
ax5.bar(number_types, number_counts, color=['#66b3ff', '#ff9999'], edgecolor='black')
ax5.set_ylabel('Count')
ax5.set_title('Number Tokenization')
ax5.grid(True, alpha=0.3, axis='y')

# 6. Tokens per word distribution
ax6 = fig.add_subplot(gs[2, :])
tokens_per_word = [d['num_tokens'] for d in frag_results['fragmentation_data']]
ax6.hist(tokens_per_word, bins=range(1, max(tokens_per_word)+2), edgecolor='black', alpha=0.7, color='lightgreen')
ax6.axvline(frag_results['avg_tokens_per_word'], color='red', linestyle='--', linewidth=2,
            label=f'Mean: {frag_results["avg_tokens_per_word"]:.2f}')
ax6.set_xlabel('Number of Tokens per Tamil Word')
ax6.set_ylabel('Frequency')
ax6.set_title('Distribution of Fragmentation Across All Tamil Words')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.suptitle(f'Tokenization Analysis Dashboard - {model_name}', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(f'tokenization_dashboard_{model_name.replace("/", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Dashboard saved as: tokenization_dashboard_{model_name.replace('/', '_')}.png")

In [ ]:
import os
from pathlib import Path
import random
from tqdm.auto import tqdm

# Configuration
WIKI_BASE_PATH = "/kaggle/input/datasets/younusmohamed/tamil-tamizh-wikipedia-articles/Tamil Wikipedia Text Articles"  # Update this to your actual path
SAMPLE_SIZE = 1000  # Number of sentences to sample (adjust based on your needs)
MAX_FILES_TO_PROCESS = 100  # Limit files to process (remove limit by setting to None)

def load_wikipedia_corpus(base_path, sample_size=1000, max_files=None):
    """
    Load Tamil Wikipedia articles from directory structure
    """
    print("=" * 80)
    print("LOADING TAMIL WIKIPEDIA CORPUS")
    print("=" * 80)
    
    all_sentences = []
    files_processed = 0
    total_chars = 0
    
    # Get all .txt files recursively
    wiki_path = Path(base_path)
    
    if not wiki_path.exists():
        raise FileNotFoundError(f"Path not found: {base_path}")
    
    txt_files = list(wiki_path.rglob("*.txt"))
    
    # Filter out LICENSE and README
    txt_files = [f for f in txt_files if f.name not in ['LICENSE', 'README.md']]
    
    print(f"Found {len(txt_files)} Wikipedia article files")
    
    if max_files:
        txt_files = txt_files[:max_files]
        print(f"Processing first {max_files} files...")
    
    # Process files with progress bar
    for file_path in tqdm(txt_files, desc="Reading files"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                
            # Split into sentences (simple approach - can be improved)
            # Split by Tamil sentence terminators: ., ?, !, newlines
            sentences = re.split(r'[.?!\n]+', content)
            
            # Clean and filter sentences
            for sent in sentences:
                sent = sent.strip()
                
                # Filter criteria:
                # - Must have Tamil characters
                # - Length between 10 and 500 characters (adjust as needed)
                # - Not too much English/Latin text
                if (len(sent) >= 10 and 
                    len(sent) <= 500 and
                    re.search(r'[\u0B80-\u0BFF]', sent)):  # Has Tamil chars
                    
                    all_sentences.append(sent)
                    total_chars += len(sent)
            
            files_processed += 1
            
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
    
    print(f"\n✓ Processed {files_processed} files")
    print(f"✓ Extracted {len(all_sentences):,} sentences")
    print(f"✓ Total characters: {total_chars:,}")
    
    # Sample if we have more than needed
    if len(all_sentences) > sample_size:
        print(f"\nSampling {sample_size} sentences from {len(all_sentences):,} total...")
        sampled = random.sample(all_sentences, sample_size)
    else:
        sampled = all_sentences
    
    return sampled

# Load the corpus
print("Loading Tamil Wikipedia corpus...\n")

try:
    tamil_corpus = load_wikipedia_corpus(
        WIKI_BASE_PATH, 
        sample_size=SAMPLE_SIZE,
        max_files=MAX_FILES_TO_PROCESS
    )
    
    print("\n" + "=" * 80)
    print("CORPUS STATISTICS")
    print("=" * 80)
    print(f"Total sentences in corpus: {len(tamil_corpus):,}")
    print(f"Total characters: {sum(len(s) for s in tamil_corpus):,}")
    print(f"Average sentence length: {np.mean([len(s) for s in tamil_corpus]):.1f} characters")
    print(f"Min sentence length: {min(len(s) for s in tamil_corpus)}")
    print(f"Max sentence length: {max(len(s) for s in tamil_corpus)}")
    
    # Analyze sentence length distribution
    lengths = [len(s) for s in tamil_corpus]
    print(f"\nSentence length percentiles:")
    print(f"  25th percentile: {np.percentile(lengths, 25):.0f} chars")
    print(f"  50th percentile (median): {np.percentile(lengths, 50):.0f} chars")
    print(f"  75th percentile: {np.percentile(lengths, 75):.0f} chars")
    print(f"  95th percentile: {np.percentile(lengths, 95):.0f} chars")
    
    print("\n" + "=" * 80)
    print("RANDOM SAMPLE OF SENTENCES:")
    print("=" * 80)
    for i, sent in enumerate(random.sample(tamil_corpus, min(5, len(tamil_corpus))), 1):
        preview = sent[:100] + "..." if len(sent) > 100 else sent
        print(f"{i}. {preview}")
    
    print("\n✓ Corpus loaded successfully!")
    
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    print("\nPlease update WIKI_BASE_PATH to point to your Tamil Wikipedia folder")
    print("Example: WIKI_BASE_PATH = '/path/to/Tamil Wikipedia Text Articles/'")
    
    # Fallback to small corpus
    print("\nFalling back to small built-in corpus for testing...")
    tamil_corpus = [
        "நான் பள்ளிக்குச் செல்கிறேன்",
        "அவன் புத்தகம் படிக்கிறான்",
        "தமிழ் மொழி உலகின் பழமையான மொழிகளில் ஒன்றாகும்",
        # ... (your original small corpus)
    ]

In [ ]:
# Analyze the quality and diversity of the corpus

print("=" * 80)
print("CORPUS QUALITY ANALYSIS")
print("=" * 80)

# 1. Content type distribution
def categorize_sentence(sent):
    """Categorize sentence by content type"""
    has_numbers = bool(re.search(r'\d', sent))
    has_english = bool(re.search(r'[a-zA-Z]', sent))
    has_math = bool(re.search(r'[+\-×÷=]', sent))
    is_question = sent.strip().endswith('?')
    
    if has_math:
        return 'Mathematical'
    elif is_question:
        return 'Question'
    elif has_numbers and not has_english:
        return 'Numeric'
    elif has_english:
        return 'Mixed Script'
    else:
        return 'Pure Tamil'

# Categorize all sentences
categories = [categorize_sentence(s) for s in tamil_corpus]
category_counts = Counter(categories)

print("\nContent Type Distribution:")
for cat, count in category_counts.most_common():
    percentage = (count / len(tamil_corpus)) * 100
    print(f"  {cat:20s}: {count:5,} ({percentage:5.1f}%)")

# 2. Vocabulary richness
all_words = []
for sent in tamil_corpus:
    all_words.extend(extract_tamil_words(sent))

unique_words = len(set(all_words))
total_words = len(all_words)

print(f"\nVocabulary Statistics:")
print(f"  Total words: {total_words:,}")
print(f"  Unique words: {unique_words:,}")
print(f"  Type-Token Ratio: {unique_words/total_words:.3f}")

# 3. Most common words
word_freq = Counter(all_words)
print(f"\nMost common words:")
for word, count in word_freq.most_common(10):
    print(f"  {word:15s}: {count:5,} occurrences")

# 4. Word length distribution
word_lengths = [len(w) for w in all_words]
print(f"\nWord Length Statistics:")
print(f"  Average word length: {np.mean(word_lengths):.1f} characters")
print(f"  Median word length: {np.median(word_lengths):.0f} characters")
print(f"  Longest word: {max(all_words, key=len)} ({len(max(all_words, key=len))} chars)")

# 5. Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sentence length distribution
axes[0, 0].hist(lengths, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_xlabel('Sentence Length (characters)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Sentence Length Distribution')
axes[0, 0].axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Word length distribution
axes[0, 1].hist(word_lengths, bins=30, edgecolor='black', alpha=0.7, color='lightcoral')
axes[0, 1].set_xlabel('Word Length (characters)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Word Length Distribution')
axes[0, 1].axvline(np.mean(word_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(word_lengths):.1f}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Content type distribution
categories_list = list(category_counts.keys())
counts_list = list(category_counts.values())
axes[1, 0].barh(categories_list, counts_list, color='lightgreen', edgecolor='black')
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Content Type Distribution')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Top words
top_words = word_freq.most_common(15)
words_list = [w[0] for w in top_words]
freq_list = [w[1] for w in top_words]
axes[1, 1].barh(words_list, freq_list, color='plum', edgecolor='black')
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].set_title('Most Common Words')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('corpus_quality_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Quality analysis complete!")
print("✓ Visualization saved as: corpus_quality_analysis.png")

In [ ]:
# For large corpora, process in batches to avoid memory issues

def calculate_token_fertility_batched(tokenizer, corpus: List[str], batch_size=100):
    """
    Token Fertility calculation with batch processing for large corpora
    """
    total_tokens = 0
    total_words = 0
    fertility_per_sentence = []
    
    # Process in batches
    for i in tqdm(range(0, len(corpus), batch_size), desc="Calculating fertility"):
        batch = corpus[i:i+batch_size]
        
        for text in batch:
            result = analyze_tokenization(tokenizer, text)
            total_tokens += result['num_tokens']
            total_words += result['num_words']
            
            if result['num_words'] > 0:
                fertility = result['num_tokens'] / result['num_words']
                fertility_per_sentence.append(fertility)
    
    avg_fertility = total_tokens / total_words if total_words > 0 else 0
    
    return {
        'average_fertility': avg_fertility,
        'total_tokens': total_tokens,
        'total_words': total_words,
        'fertility_per_sentence': fertility_per_sentence,
        'min_fertility': min(fertility_per_sentence) if fertility_per_sentence else 0,
        'max_fertility': max(fertility_per_sentence) if fertility_per_sentence else 0,
        'std_fertility': np.std(fertility_per_sentence) if fertility_per_sentence else 0,
        'median_fertility': np.median(fertility_per_sentence) if fertility_per_sentence else 0
    }

# Calculate with progress bar
print("Calculating Token Fertility on Wikipedia corpus...")
fertility_results = calculate_token_fertility_batched(tokenizer, tamil_corpus, batch_size=100)

print("\n" + "=" * 80)
print("TOKEN FERTILITY ANALYSIS (Wikipedia Corpus)")
print("=" * 80)
print(f"\nAverage Token Fertility: {fertility_results['average_fertility']:.3f}")
print(f"Median Token Fertility: {fertility_results['median_fertility']:.3f}")
print(f"Interpretation: Each Tamil word → {fertility_results['average_fertility']:.2f} tokens (average)")
print(f"\nTotal tokens generated: {fertility_results['total_tokens']:,}")
print(f"Total Tamil words: {fertility_results['total_words']:,}")
print(f"\nRange: {fertility_results['min_fertility']:.2f} - {fertility_results['max_fertility']:.2f}")
print(f"Standard deviation: {fertility_results['std_fertility']:.3f}")

# Enhanced visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(fertility_results['fertility_per_sentence'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(fertility_results['average_fertility'], color='red', linestyle='--', linewidth=2,
                label=f'Mean: {fertility_results["average_fertility"]:.2f}')
axes[0].axvline(fertility_results['median_fertility'], color='green', linestyle='--', linewidth=2,
                label=f'Median: {fertility_results["median_fertility"]:.2f}')
axes[0].set_xlabel('Token Fertility (tokens per word)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Token Fertility Distribution (Wikipedia)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(fertility_results['fertility_per_sentence'], vert=True)
axes[1].set_ylabel('Token Fertility')
axes[1].set_title('Token Fertility Box Plot')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fertility_wikipedia.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compare Wikipedia corpus results with expectations

print("=" * 80)
print("WIKIPEDIA CORPUS: KEY INSIGHTS")
print("=" * 80)

# Token efficiency
tokens_per_char = fertility_results['total_tokens'] / sum(len(s) for s in tamil_corpus)
print(f"\nTokenization Efficiency:")
print(f"  Tokens per character: {tokens_per_char:.3f}")
print(f"  Characters per token: {1/tokens_per_char:.3f}")

# Compression ratio
original_size = sum(len(s.encode('utf-8')) for s in tamil_corpus)
token_size = fertility_results['total_tokens'] * 4  # Assume 4 bytes per token ID
compression_ratio = original_size / token_size

print(f"\nCompression Analysis:")
print(f"  Original size (UTF-8): {original_size:,} bytes")
print(f"  Tokenized size (IDs): {token_size:,} bytes")
print(f"  Compression ratio: {compression_ratio:.2f}x")

# Fragmentation severity
severe_fragmentation = sum(1 for f in fertility_results['fertility_per_sentence'] if f > 3.0)
severe_pct = (severe_fragmentation / len(fertility_results['fertility_per_sentence'])) * 100

print(f"\nFragmentation Severity:")
print(f"  Sentences with >3 tokens/word: {severe_fragmentation:,} ({severe_pct:.1f}%)")

if severe_pct > 20:
    print(f"  ⚠️ HIGH: {severe_pct:.0f}% of sentences heavily fragmented")
elif severe_pct > 10:
    print(f"  ⚠️ MODERATE: {severe_pct:.0f}% of sentences moderately fragmented")
else:
    print(f"  ✓ LOW: Only {severe_pct:.0f}% heavily fragmented")

# Estimate reasoning impact
print(f"\nEstimated Impact on Reasoning:")
if fertility_results['average_fertility'] > 2.5:
    print(f"  ⚠️ Token fertility ({fertility_results['average_fertility']:.2f}) may hurt reasoning performance")
    print(f"     - More tokens = longer sequences = harder attention")
    print(f"     - Fragmented words = broken semantic units")
else:
    print(f"  ✓ Token fertility ({fertility_results['average_fertility']:.2f}) is acceptable")

# Save comprehensive report
report = {
    'model': model_name,
    'corpus_size': len(tamil_corpus),
    'total_tokens': fertility_results['total_tokens'],
    'total_words': fertility_results['total_words'],
    'fertility': {
        'mean': float(fertility_results['average_fertility']),
        'median': float(fertility_results['median_fertility']),
        'std': float(fertility_results['std_fertility']),
        'min': float(fertility_results['min_fertility']),
        'max': float(fertility_results['max_fertility'])
    },
    'efficiency': {
        'tokens_per_char': float(tokens_per_char),
        'compression_ratio': float(compression_ratio)
    },
    'fragmentation': {
        'severe_count': int(severe_fragmentation),
        'severe_percentage': float(severe_pct)
    }
}

with open('wikipedia_tokenization_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n✓ Comprehensive report saved to: wikipedia_tokenization_report.json")